# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [2]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_are

def read_tsv_matrix(path):
    df = pd.read_csv(path, sep="\t")
    if df.columns[0].lower() in ("index", "row", "state"):
        df = df.drop(columns=[df.columns[0]])
    return df.to_numpy(dtype=float)

A = read_tsv_matrix("model-A.tsv")
B = read_tsv_matrix("model-B.tsv")
Q = read_tsv_matrix("cost-Q.tsv")
R = read_tsv_matrix("cost-R.tsv")

P = solve_discrete_are(A, B, Q, R)
K = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)

Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [ ]:
# YOUR CHANGES HERE

out = pd.DataFrame([{"x": K[0, 0], "v": K[0, 1], "a": K[0, 2]}])
out.to_csv("control-K-intended.tsv", sep="\t", index=False)


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [4]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

K_df = pd.read_csv("control-K-intended.tsv", sep="\t")
K = K_df[["x", "v", "a"]].to_numpy(dtype=float)
if K.shape != (1, 3):
    raise ValueError(f"Expected K shape (1,3), got {K.shape}")

log = pd.read_csv("qc-log.tsv", sep="\t")

xva_cols = ["x", "v", "a"]
if not all(c in log.columns for c in xva_cols):
    raise ValueError(f"qc-log.tsv must contain columns {xva_cols}. Found: {list(log.columns)}")

X = log[xva_cols].to_numpy(dtype=float)
u_check = -(X @ K.T).reshape(-1)

idx = log["index"].to_numpy() if "index" in log.columns else np.arange(len(log))



Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [5]:
# YOUR CHANGES HERE
out = pd.DataFrame({"index": idx, "u_check": u_check})
out.to_csv("qc-check.tsv", sep="\t", index=False)
...

Ellipsis

Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [6]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

log = pd.read_csv("qc-log.tsv", sep="\t")

X = log[["x", "v", "a"]].to_numpy(dtype=float)

if "u" in log.columns:
    u = log["u"].to_numpy(dtype=float)
elif "action" in log.columns:
    u = log["action"].to_numpy(dtype=float)
else:
    raise ValueError(f"qc-log.tsv must contain 'u' (or 'action') column. Found: {list(log.columns)}")

u = u.reshape(-1, 1)

X_design = np.hstack([X, np.ones((X.shape[0], 1))])
coef, _, _, _ = np.linalg.lstsq(X_design, u, rcond=None)
k = coef[:3, 0]



Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [7]:
# YOUR CHANGES HERE

out = pd.DataFrame([{"x": -k[0], "v": -k[1], "a": -k[2]}])
out.to_csv("control-K-actual.tsv", sep="\t", index=False)

Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [8]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

train = pd.read_csv("qc-train.tsv", sep="\t")

required_cols = ["x", "v", "a", "u"]
for c in required_cols:
    if c not in train.columns:
        raise ValueError(f"qc-train.tsv must contain columns {required_cols}. Found: {list(train.columns)}")

if all(c in train.columns for c in ["x_next", "v_next", "a_next"]):
    X_t = train[["x", "v", "a"]].to_numpy(dtype=float)
    U_t = train[["u"]].to_numpy(dtype=float)
    X_next = train[["x_next", "v_next", "a_next"]].to_numpy(dtype=float)
else:
    X_t = train[["x", "v", "a"]].to_numpy(dtype=float)[:-1]
    U_t = train[["u"]].to_numpy(dtype=float)[:-1]
    X_next = train[["x", "v", "a"]].to_numpy(dtype=float)[1:]

Z = np.hstack([X_t, U_t])
theta, _, _, _ = np.linalg.lstsq(Z, X_next, rcond=None)





Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [9]:
# YOUR CHANGES HERE
A_new = theta[:3, :].T
B_new = theta[3:, :].T
pd.DataFrame(A_new).to_csv("model-A-new.tsv", sep="\t", index=False, header=False)
pd.DataFrame(B_new).to_csv("model-B-new.tsv", sep="\t", index=False, header=False)

Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.